# D1c 학습 루프와 MNIST — 실습 (W3, D1 3부작 완결)

> Colab에서 **런타임 → 런타임 유형 변경 → GPU** 로 설정하면 빠릅니다(CPU도 동작, 몇 분 소요).
> 위에서부터 한 셀씩 `Shift+Enter`로 실행하세요. `___` 빈칸은 직접 채웁니다.

**이 실습이 끝나면**
1. `nn.Module`로 모델을 포장하고 파라미터 109,386개를 검산한다
2. DataLoader 배치의 shape을 읽는다 — (64, 1, 28, 28)
3. **학습 루프 5단계를 직접 완성**해 MNIST를 3 epoch 학습한다
4. `eval` + `no_grad`로 평가한다 — **정확도 ~97%**
5. 곡선·혼동행렬·틀린 예측으로 모델을 **진단**한다

**7단계 멘탈모델 초점:** 일곱 단계 전부 (3부작 완주)

## Part A. MNIST 로드 — 데이터를 눈으로 먼저
손글씨 70,000장 (train 60,000 / test 10,000). `ToTensor()`가 픽셀 0~255를 0~1로.

In [ ]:
import torch                                            # PyTorch
import torch.nn as nn                                   # 신경망 모듈
import matplotlib.pyplot as plt                         # 그래프
from torchvision import datasets, transforms            # 영상 데이터셋·변환
from torch.utils.data import DataLoader                 # 배치 공급기

device = 'cuda' if torch.cuda.is_available() else 'cpu' # 관용구(D1a)
torch.manual_seed(0)                                    # 재현성

tf = transforms.ToTensor()                              # 0~255 → 0~1 텐서
train_ds = datasets.MNIST('./data', train=True,  download=True, transform=tf)  # 학습 6만
test_ds  = datasets.MNIST('./data', train=False, download=True, transform=tf)  # 시험 1만
print('train:', len(train_ds), '| test:', len(test_ds)) # 60000 / 10000
img0, label0 = train_ds[0]                              # 첫 샘플 꺼내기(Dataset[i])
print('한 장 shape:', img0.shape, '| 정답:', label0)    # (1,28,28) — 채널,높이,너비(D1a)

fig, axes = plt.subplots(2, 6, figsize=(10, 3.6))       # 샘플 12장 보기
for i, ax in enumerate(axes.flat):
    img, lab = train_ds[i]                              # i번째 샘플
    ax.imshow(img[0], cmap='gray')                      # 채널 차원 제거해 표시
    ax.set_title(f'label {lab}')                        # 정답(영어)
    ax.axis('off')                                      # 축 끄기
plt.tight_layout(); plt.show()                          # 데이터를 눈으로 먼저 — 항상!

## Part B. DataLoader — 배치 공급기
왜 배치인가: ①메모리 ②더 자주 갱신(1 epoch ≈ 938걸음) ③일반화. **shuffle은 학습에만.**

In [ ]:
train_loader = DataLoader(train_ds, batch_size=64, shuffle=___)  # ✍️ 빈칸: 학습은 섞는다
test_loader  = DataLoader(test_ds,  batch_size=256)     # 평가는 안 섞어도 됨

xb, yb = next(iter(train_loader))                       # 배치 하나 꺼내 보기
print('배치 이미지:', xb.shape)                         # (64,1,28,28) — 배치,채널,높이,너비
print('배치 정답  :', yb.shape, '| 앞 8개:', yb[:8].tolist())  # (64,)
print('1 epoch =', len(train_loader), 'step')           # 60000/64 → 938걸음

## Part C. nn.Module — 모델을 클래스로 포장 ⭐
D1b에선 가중치 텐서를 손으로 관리했습니다. 이제 nn.Module이 대신합니다.
출력층은 **10개(클래스 수) logit** — softmax 없음(D1b: CrossEntropyLoss가 포함).

In [ ]:
class MLP(nn.Module):                                   # nn.Module 상속 = 포장 규격
    def __init__(self):
        super().__init__()                              # 부모 초기화(필수 관용구)
        self.net = nn.Sequential(                       # 층을 순서대로
            nn.Flatten(),                               # (배치,1,28,28) → (배치,784)
            nn.Linear(28*28, 128), nn.ReLU(),           # 은닉1 + 비선형
            nn.Linear(128, 64), nn.ReLU(),              # 은닉2 + 비선형
            nn.Linear(64, ___)                          # ✍️ 빈칸: 출력 = 클래스 개수(logit)
        )
    def forward(self, x):                               # 순전파 정의
        return self.net(x)                              # model(x)가 이걸 호출

model = MLP().to(device)                                # 만들고 장치로
print(model)                                            # 구조 확인
n_params = sum(p.___() for p in model.parameters())     # ✍️ 빈칸: 텐서의 원소 개수 메서드
print('파라미터 수:', n_params)                         # 109386 — 손 계산과 일치(설명서 §3)

## Part D. 학습 루프 5단계 ⭐⭐ — 3부작의 심장
①배치 → ②zero_grad → ③순전파 → ④손실 → ⑤backward+step. **전부 D1a·D1b 부품입니다.**

> epoch마다 시험 정확도도 기록합니다(진단용 — 평가 코드는 Part E에서 직접 써 봅니다).

In [ ]:
criterion = nn.CrossEntropyLoss()                       # 분류 손실(logit 입력! — D1b)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)  # Adam(D1b) — 목록은 자동 수집
EPOCHS = 3                                              # 전체 3바퀴

def test_acc():                                         # 진단용 미리보기(직접 작성은 Part E)
    model.eval()                                        # 평가 모드
    correct = total = 0
    with torch.no_grad():                               # 기록 끄기
        for xb, yb in test_loader:
            pred = model(xb.to(device)).argmax(1).cpu() # 최고 점수 클래스
            correct += (pred == yb).sum().item(); total += yb.size(0)
    model.train()                                       # 학습 모드로 복귀(중요!)
    return correct / total

train_losses, test_accs = [], []                        # epoch별 기록
for epoch in range(EPOCHS):
    model.train()                                       # 학습 모드
    running = 0.0
    for xb, yb in train_loader:                         # ① 배치 꺼내기
        xb, yb = xb.to(device), yb.to(device)           #    데이터도 장치로(D1a)
        optimizer.zero_grad()                           # ② 기울기 초기화(D1b 규칙①)
        pred = model(xb)                                # ③ 순전파(D1a)
        loss = criterion(pred, yb)                      # ④ 손실(D1b)
        loss.___()                                      # ✍️ 빈칸: ⑤a 역전파
        optimizer.___()                                 # ✍️ 빈칸: ⑤b 가중치 갱신
        running += loss.item()                          # 손실 누적
    train_losses.append(running / len(train_loader))    # epoch 평균 손실
    test_accs.append(test_acc())                        # epoch별 시험 정확도
    print(f'epoch {epoch+1}: train loss = {train_losses[-1]:.4f} | test acc = {test_accs[-1]:.4f}')

## Part E. 평가 — 스위치 두 개를 직접 쓰기
`model.eval()` = 학습 전용 동작(dropout 등) 끄기 / `torch.no_grad()` = 기울기 기록 끄기. **역할이 달라 둘 다.**

In [ ]:
model.___()                                             # ✍️ 빈칸: 평가 모드 전환
correct = total = 0
with torch.___():                                       # ✍️ 빈칸: 기울기 기록 끄기
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)           # 장치로
        pred = model(xb).argmax(1)                      # 최고 점수 클래스 = 예측
        correct += (pred == yb).sum().item()            # 맞은 개수
        total += yb.size(0)                             # 전체 개수
print('최종 test accuracy:', round(correct / total, 4)) # ~0.97 — 첫 신경망 완성!

## Part F. 진단 ① — 곡선 읽기 (M2 원칙: train과 test를 함께)
train 손실↓ 계속인데 test 정확도가 꺾이면 **과적합** 신호입니다.

In [ ]:
fig, ax1 = plt.subplots(figsize=(6, 4))                 # 이중 축 그래프
ep = range(1, EPOCHS + 1)                               # epoch 축
ax1.plot(ep, train_losses, 'o-', color='tab:blue', label='train loss')  # 손실
ax1.set_xlabel('epoch'); ax1.set_ylabel('train loss', color='tab:blue')  # 축(영어)
ax1.set_xticks(list(ep))                                # 정수 눈금
ax2 = ax1.twinx()                                       # 오른쪽 축
ax2.plot(ep, test_accs, 's--', color='tab:red', label='test accuracy')  # 정확도
ax2.set_ylabel('test accuracy', color='tab:red')        # 축(영어)
plt.title('Training curves: loss down, accuracy up')    # 제목(영어)
plt.grid(True); plt.show()                              # 둘을 함께 보는 습관(M2)

## Part G. 진단 ② — 혼동행렬 (M2 연결)
행=실제, 열=예측. 대각선 밖 밝은 칸 = 자주 헷갈리는 쌍 (MNIST 단골: 4↔9, 3↔5, 7↔2).

In [ ]:
cm = torch.zeros(10, 10, dtype=torch.int32)             # 10x10 혼동행렬
with torch.no_grad():                                   # 평가이므로 기록 끄기
    for xb, yb in test_loader:
        preds = model(xb.to(device)).argmax(dim=___).cpu()  # ✍️ 빈칸: 클래스 차원(D1a dim!)
        for t, p in zip(yb, preds):                     # 실제 t, 예측 p
            cm[t, p] += 1                               # 해당 칸 +1

plt.figure(figsize=(5.5, 5))                            # 그리기
plt.imshow(cm, cmap='Blues')                            # 진할수록 많음
plt.colorbar(); plt.xticks(range(10)); plt.yticks(range(10))  # 눈금
plt.xlabel('predicted'); plt.ylabel('true')             # 축(영어)
plt.title('Confusion matrix (test 10,000)')             # 제목(영어)
plt.show()                                              # 대각선 = 정답

off = cm.clone(); off.fill_diagonal_(0)                 # 대각선 제거
vals, idxs = off.flatten().topk(3)                      # 가장 헷갈린 쌍 top3
for v, i in zip(vals, idxs):
    print(f'실제 {i.item() // 10} → 예측 {i.item() % 10} : {v.item()}회')  # 약점 지도

## Part H. 진단 ③ — 틀린 예측을 눈으로
납득되는 실수(악필)인가, 이상한 실수(버그 의심)인가 — 에러 분석 습관.

In [ ]:
wrong_img, wrong_t, wrong_p = [], [], []                # 틀린 사례 수집
with torch.no_grad():
    for xb, yb in test_loader:
        preds = model(xb.to(device)).argmax(1).cpu()    # 예측
        for img, t, p in zip(xb, yb, preds):
            if t != p and len(wrong_img) < 9:           # 틀린 것만 9장까지
                wrong_img.append(img[0]); wrong_t.append(t.item()); wrong_p.append(p.item())
        if len(wrong_img) >= 9: break                   # 충분하면 중단

fig, axes = plt.subplots(3, 3, figsize=(6, 6.5))        # 3x3 그리드
for i, ax in enumerate(axes.flat):
    ax.imshow(wrong_img[i], cmap='gray')                # 틀린 이미지
    ax.set_title(f'true {wrong_t[i]} / pred {wrong_p[i]}')  # 정답/예측(영어)
    ax.axis('off')                                      # 축 끄기
plt.tight_layout(); plt.show()                          # 납득되는 실수인가?

## 🤖 AI 코파일럿 활용 (선택) — ai-native v1
막히면 AI 튜터에게 묻되, **먼저 스스로 생각**하고 답을 **실행으로 검증**하세요.

**좋은 질문 예시**
- "학습 루프 5단계를 내가 순서대로 쓸 테니, 각 단계가 D1a/D1b의 무엇인지 채점해 줘."
- "`model.eval()`과 `torch.no_grad()`의 역할 차이를 내가 설명해 볼게. 틀린 곳을 질문으로 짚어 줘."
- "내 혼동행렬에서 4→9가 가장 밝아. 개선 아이디어를 내가 3개 낼 테니 평가해 줘."
- "state_dict에 구조가 저장되지 않는 이유를 물어봐 줘. 내가 먼저 답할게."

**가드레일**
1. 먼저 손으로 생각 → 그 다음 AI
2. AI 코드는 *왜 그런지* 설명할 수 있을 때만 사용
3. AI 출력은 실행으로 검증

## 정리 & 자가 점검 — D1 3부작 완결 🎉

**오늘 한 일 3줄**
1. nn.Module로 모델을 포장하고(파라미터 109,386 검산), DataLoader로 배치 공급(1 epoch=938 step)
2. 학습 루프 5단계를 직접 완성해 MNIST 3 epoch 학습 → 정확도 ~97%
3. 곡선·혼동행렬·틀린 예측으로 진단 — 7단계 멘탈모델 한 바퀴 완주

**스스로 점검**
- [ ] 학습 루프 5단계를 외워서 쓸 수 있다 (D2부터 매주 쓴다!)
- [ ] (64,1,28,28) → Flatten → (64,784)의 shape 흐름을 안다
- [ ] eval()과 no_grad()의 역할 차이를 설명할 수 있다
- [ ] 혼동행렬에서 모델의 약점(헷갈리는 쌍)을 읽을 수 있다
- [ ] train↓ + test↓ 조합이 과적합 신호임을 안다(M2)

**🔹심화 (선택)**
- **dropout 실험:** `nn.Linear(128, 64), nn.ReLU()` 뒤에 `nn.Dropout(0.3),`을 넣고 재학습 — 3 epoch에선 차이가 작지만 EPOCHS=10으로 늘려 train/test 곡선을 비교해 보세요(과적합 완화 관찰).
- **저장/불러오기:** `torch.save(model.state_dict(), 'mnist_mlp.pt')` → 새 `MLP()`에 `load_state_dict` → 같은 정확도가 나오는지 확인.
- **손잡이 실험:** 은닉 (128,64)→(256,128), batch_size 64→256, lr 1e-3→1e-2 — 각각 정확도·속도가 어떻게 변하는지 표로 정리.